# Track 1: Reasoned Financial Sentiment Fine-Tuning
This notebook demonstrates fine-tuning the **Gemma 4 E2B-IT** model on the `lmassaron/FinancialPhraseBank_explained` dataset using **TRL** and **LoRA** (4-bit QLoRA).

### Objectives
- Learn how to inject Chain-of-Thought (CoT) reasoning into a classification task.
- Perform Supervised Fine-Tuning (SFT) using Hugging Face `trl` and `peft` libraries.
- Optimize training parameters to fit within a 16GB VRAM constraint.

In [ ]:
# Install extra dependencies if needed (e.g. on Google Colab)
# %pip install -U transformers trl peft accelerate bitsandbytes datasets


In [ ]:
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Check device and capability
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
print(f"Using device: {device} | Dtype: {compute_dtype}")

## 1. Load Dataset
We pull the `lmassaron/FinancialPhraseBank_explained` dataset which contains news sentences, ground-truth sentiments, and human-like explanations.

In [ ]:
DATASET_ID = "lmassaron/FinancialPhraseBank_explained"
print(f"Loading {DATASET_ID}...")
dataset = load_dataset(DATASET_ID)

# Rename columns to standard names
dataset = dataset.rename_columns({"sentence": "text", "explanation": "reasoning"})
train_ds = dataset["train"]
eval_ds = dataset["validation"]
print(train_ds)
print("Sample sentence:", train_ds[0]["text"])
print("Sample sentiment:", train_ds[0]["sentiment"])
print("Sample explanation:", train_ds[0]["reasoning"])

## 2. Formatting Prompts using Chat Template
We structure the news headline, instructions, and target sentiment/explanations into a chat template using Gemma 4's format.

In [ ]:
SYSTEM_PROMPT = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)

MODEL_ID = "google/gemma-4-E2B-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_prompt(example):
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + f'\nHeadline: "{example["text"]}"'},
        {
            "role": "assistant",
            "content": f"<sentiment>{example['sentiment']}</sentiment>\n<reasoning>{example['reasoning']}</reasoning>",
        },
    ]
    # Format without tokenizing so SFTTrainer can tokenize it
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": prompt}


train_mapped = train_ds.map(format_prompt)
eval_mapped = eval_ds.map(format_prompt)
print("Mapped Prompt Preview:\n", train_mapped[0]["text"])

## 3. Load Model with 4-bit Quantization
We load the base `google/gemma-4-E2B-it` model in 4-bit precision to fit within the 16GB VRAM constraint, and configure LoRA adapters targeting all linear modules.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config, device_map="auto"
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 4. Run SFT Trainer
We configure the Hugging Face `trl` SFT Config and start fine-tuning. This is optimized for a batch size of 2 and gradient accumulation of 4 (effective batch size of 8).

In [ ]:
training_args = SFTConfig(
    output_dir="gemma4-sentiment-lora",
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    lr_scheduler_type="cosine",
    warmup_steps=0,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    report_to="none",
    dataset_kwargs={
        "add_special_tokens": False,
    },
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# Disable KV cache during training to save VRAM
model.config.use_cache = False

trainer.train()

# Save the fine-tuned adapter
trainer.model.save_pretrained("gemma4-sentiment-lora-adapter")
tokenizer.save_pretrained("gemma4-sentiment-lora-adapter")
print("Adapter successfully saved!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_training_metrics(
    log_history, 
    window=25, 
    save_path="training_curves.png", 
    figsize=(15, 5), 
    dpi=150
):
    """Extracts and plots training loss, validation loss, and gradient norm with a moving average"""

    logs = pd.DataFrame(log_history)
    
    if "step" not in logs.columns:
        raise ValueError("Log history must contain a 'step' column.")
        
    logs = logs[logs["step"].notna()].copy()
    
    # Create a 1x3 grid for the 3 metrics
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # 1. Training Loss
    if "loss" in logs:
        train_logs = logs[logs["loss"].notna()].copy()
        train_logs["loss_ma"] = train_logs["loss"].rolling(window=window, min_periods=1).mean()
        axes[0].plot(train_logs["step"], train_logs["loss"], alpha=0.3, label="Raw Train Loss")
        axes[0].plot(train_logs["step"], train_logs["loss_ma"], color="blue", label=f"MA ({window})")
        axes[0].set_title("Training Loss")
        axes[0].legend()

    # 2. Validation Loss
    if "eval_loss" in logs:
        eval_logs = logs[logs["eval_loss"].notna()].copy()
        if not eval_logs.empty:
            eval_logs["eval_loss_ma"] = eval_logs["eval_loss"].rolling(window=window, min_periods=1).mean()
            # Added markers because eval steps are usually logged less frequently
            axes[1].plot(eval_logs["step"], eval_logs["eval_loss"], alpha=0.4, marker='o', label="Raw Eval Loss")
            axes[1].plot(eval_logs["step"], eval_logs["eval_loss_ma"], color="orange", label=f"MA ({window})")
            axes[1].set_title("Validation Loss")
            axes[1].legend()
        else:
            axes[1].set_title("Validation Loss (No Data)")

    # 3. Gradient Norm
    if "grad_norm" in logs:
        grad_logs = logs[logs["grad_norm"].notna()].copy()
        grad_logs["grad_norm_ma"] = grad_logs["grad_norm"].rolling(window=window, min_periods=1).mean()
        axes[2].plot(grad_logs["step"], grad_logs["grad_norm"], alpha=0.3, label="Raw Grad Norm")
        axes[2].plot(grad_logs["step"], grad_logs["grad_norm_ma"], color="red", label=f"MA ({window})")
        axes[2].set_title("Gradient Norm")
        axes[2].legend()

    # Formatting
    for ax in axes.flat:
        ax.set_xlabel("Step")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi)
        
    plt.show()
    
    return fig, axes


plot_training_metrics(trainer.state.log_history)

## 5. Evaluation and Inference
We load the fine-tuned adapter, merge it with the base model, and run batch evaluation on the original (non-augmented) test set Headlines. We measure accuracy, tag format integrity, and output a detailed classification report.

In [ ]:
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import re
import pandas as pd

# 1. Load fine-tuned model
print("Loading merged base model and adapter...")
eval_model = PeftModel.from_pretrained(model, "gemma4-sentiment-lora-adapter")
eval_model = eval_model.eval()

# 2. Prepare test data (filtering out augmented samples for clean evaluation)
test_df = dataset["test"].to_pandas()
if "is_augmented" in test_df.columns:
    test_df = test_df[test_df["is_augmented"] == False].reset_index(drop=True)
print(f"Evaluating on {len(test_df)} original test headlines.")

# 3. Configure tokenizer for batch left-padding
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def extract_tag(text, tag):
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None


results = []
batch_size = 16

for start in tqdm(range(0, len(test_df), batch_size), desc="Evaluating"):
    batch_df = test_df.iloc[start : start + batch_size]
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": SYSTEM_PROMPT + f'Headline: "{row["text"]}"'}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for _, row in batch_df.iterrows()
    ]

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    for i, raw_output in enumerate(decoded):
        sentiment_tag = extract_tag(raw_output, "sentiment")
        reasoning_pred = extract_tag(raw_output, "reasoning")
        results.append(
            {
                "sentiment_true": batch_df.iloc[i]["sentiment"],
                "sentiment_pred": sentiment_tag.lower() if sentiment_tag else "none",
                "reasoning_pred": reasoning_pred if reasoning_pred else "none",
                "tag_integrity": sentiment_tag is not None
                and reasoning_pred is not None,
            }
        )

results_df = pd.DataFrame(results)
accuracy = accuracy_score(results_df["sentiment_true"], results_df["sentiment_pred"])
integrity = results_df["tag_integrity"].mean()
print(f"\nAccuracy: {accuracy:.2%}")
print(f"Tag Integrity: {integrity:.2%}")
print("\nClassification Report:")
print(classification_report(results_df["sentiment_true"], results_df["sentiment_pred"]))